In [2]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' # Only errors are logged
os.environ['TF_GPU_ALLOCATOR'] ='cuda_malloc_async'

import numpy as np
import keras
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import layers
from keras import ops

# TF imports related to tf.data preprocessing
from tensorflow import data as tf_data
from tensorflow import image as tf_image
from tensorflow.keras.utils import plot_model

keras.utils.set_random_seed(42)
from sklearn.model_selection import train_test_split

I0000 00:00:1776925646.311325 2008414 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776925647.375811 2008414 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [ ]:
BATCH_SIZE = 32
NUM_CLASSES = 8
EPOCHS = 100
SAMPLE_RATE = 16000
OUT_SEQ_LEN = 72000

In [ ]:
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(directory='dataset_mel_img',
image_size=(300, 300),
subset='both',
batch_size=BATCH_SIZE,
validation_split=0.2,
seed=42)

Found 4240 files belonging to 8 classes.
Using 3392 files for training.


: 

In [ ]:
test_ds = val_ds.shard(num_shards=2, index=0)
val_ds = val_ds.shard(num_shards=2, index=1)

In [ ]:
incv3 = keras.applications.InceptionV3(weights=None,include_top=False,input_shape=(299,299,3))
model = keras.Sequential([
  layers.Input(shape=(28,28,1),name='input'),
  layers.Resizing(299,299),
#   tf.keras.layers.Lambda(tf.image.grayscale_to_rgb),
  incv3,
  # layers.GlobalAveragePooling2D(name='gp'),
  layers.Flatten(),
  layers.Dense((1024),activation='relu'),
  layers.Dense((512),activation='relu'),
  layers.Dense((NUM_CLASSES),activation = 'softmax',name='output')
])

In [ ]:
model.summary()

In [ ]:
keras.backend.clear_session(free_memory=True)
model.compile(
    optimizer=keras.optimizers.AdamW(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy()],
)


In [ ]:
history = model.fit(train_ds,
          batch_size=BATCH_SIZE,
          epochs=EPOCHS,
          validation_data=val_ds,
        #   callbacks=callbacks
          )

In [ ]:
plt.plot(history.history['sparse_categorical_accuracy'])
plt.plot(history.history['val_sparse_categorical_accuracy'],linestyle='--')
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['Training', 'Validation'], loc='lower right')
plt.show()

In [ ]:
model.evaluate(test_ds)